# 54 — Resume Ranking
**Goal:** Rank multiple resumes against a job description for candidate shortlisting.

## 1. Batch Resume Ranking

In [ ]:
class ResumeRanker:
    def __init__(self):
        self.matcher = ResumeJDMatcher() if 'ResumeJDMatcher' in dir() else None
    
    def rank(self, resumes, jd_text):
        results = []
        for i, (name, text) in enumerate(resumes):
            if self.matcher:
                match = self.matcher.match(text, jd_text)
                score = match['score']
                details = match['details']
            else:
                # Simplified scoring
                skills = ["Python", "NLP", "TensorFlow", "SQL", "AWS", "Docker"]
                jd_skills = [s for s in skills if s.lower() in jd_text.lower()]
                resume_skills = [s for s in skills if s.lower() in text.lower()]
                skill_ratio = len([s for s in jd_skills if s in resume_skills]) / max(len(jd_skills), 1)
                score = skill_ratio * 80 + 10
                details = {"skill_match": skill_ratio, "embedding": 0.5}
            
            results.append({
                "name": name, "score": round(score * 100, 1) if score <= 1 else round(score, 1),
                "skills_found": len(resume_skills) if 'resume_skills' in dir() else 0,
            })
        
        results.sort(key=lambda x: x["score"], reverse=True)
        return results

resumes = [
    ("Alice", "Senior data scientist, Python, NLP, TensorFlow, 5 years"),
    ("Bob", "Java backend developer, Spring Boot, microservices, 3 years"),
    ("Charlie", "Data engineer, Python, SQL, Spark, AWS, 4 years"),
    ("Diana", "NLP researcher, Python, PyTorch, Transformers, BERT, PhD"),
]
jd = "Senior Data Scientist: Python, NLP, TensorFlow required, 5+ years"

ranker = ResumeRanker()
ranked = ranker.rank(resumes, jd)
print(f"Ranking for: {jd}")
print("=" * 50)
for i, r in enumerate(ranked):
    print(f"  #{i+1} {r['name']:10s} | Score: {r['score']:.1f} | Skills: {r['skills_found']}")

## 2. Display Results

In [ ]:
# Display as table
print(f"\\n{'Rank':<6}{'Name':<12}{'Score':<10}{'Status':<12}")
print("-" * 40)
for i, r in enumerate(ranked):
    if r['score'] >= 80:
        status = "SHORTLIST"
    elif r['score'] >= 60:
        status = "MAYBE"
    else:
        status = "PASS"
    print(f"{'#'+str(i+1):<6}{r['name']:<12}{r['score']:<10}{status:<12}")

## Summary: Resume ranking aggregates all ATS dimensions into a single sortable score.